# 01 · VAERS — Exploración y perfilado inicial

**Proyecto:** ¿Por qué los conteos crudos de VAERS engañan?

Este notebook hace lo mínimo indispensable y bien hecho: **carga** los datos de VAERS,
entiende su **estructura y grano**, revisa su **calidad**, y deja preparado el **denominador**
para el análisis de tasas. No saca conclusiones todavía — primero hay que conocer el dato.

> Fuente y limitaciones documentadas en `DATA_SOURCES.md`. Recordatorio clave: VAERS es
> vigilancia **pasiva**, los reportes **no están verificados** y **no implican causalidad**.

## 1. Conseguir los datos

VAERS **no** se puede descargar con `pd.read_csv(url)` directo: la descarga exige aceptar
los términos en la web. El camino reproducible y honesto es:

1. Ve a **https://vaers.hhs.gov/data/datasets.html**
2. Acepta los términos y descarga el ZIP del año que quieras analizar (ej. `2021VAERSData.zip`).
   *2021 es el año más rico para vacunas COVID.*
3. Descomprime: obtendrás `2021VAERSDATA.csv`, `2021VAERSVAX.csv`, `2021VAERSSYMPTOMS.csv`.
4. Súbelos a Colab con la celda de abajo (o móntalos desde Google Drive).

Ejecuta esta celda y selecciona los 3 CSV:

In [ ]:
# Subir los 3 CSV de VAERS a la sesión de Colab.
# (En Colab el sistema de archivos es efímero: se borra al cerrar. Por eso subimos cada vez,
#  lo que cumple la regla de oro: los datos crudos siempre se regeneran desde la fuente.)
try:
    from google.colab import files
    uploaded = files.upload()   # selecciona 2021VAERSDATA.csv, 2021VAERSVAX.csv, 2021VAERSSYMPTOMS.csv
except ImportError:
    print("No estás en Colab. Coloca los CSV en data/raw/ y ajusta las rutas abajo.")


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)

YEAR = "2021"   # cambia el año si descargaste otro

# Rutas: en Colab los archivos subidos quedan en el directorio actual.
paths = {
    "data":     f"{YEAR}VAERSDATA.csv",
    "vax":      f"{YEAR}VAERSVAX.csv",
    "symptoms": f"{YEAR}VAERSSYMPTOMS.csv",
}


## 2. Cargar los 3 archivos

**Gotcha real:** los CSV de VAERS están en **Latin-1 (ISO-8859-1)**, no UTF-8. Si los cargas
con el encoding por defecto de pandas, revienta con `UnicodeDecodeError`. Por eso forzamos
`encoding="latin-1"`.

In [ ]:
def cargar_vaers(path):
    # low_memory=False evita warnings de tipos mixtos; encoding latin-1 es obligatorio.
    return pd.read_csv(path, encoding="latin-1", low_memory=False)

df_data     = cargar_vaers(paths["data"])
df_vax      = cargar_vaers(paths["vax"])
df_symptoms = cargar_vaers(paths["symptoms"])

for nombre, df in [("DATA", df_data), ("VAX", df_vax), ("SYMPTOMS", df_symptoms)]:
    print(f"{nombre:9s} -> {df.shape[0]:>8,} filas x {df.shape[1]:>3} columnas")


## 3. Estructura y grano

El **grano** (qué representa cada fila) es lo primero que hay que entender, porque determina
cómo se pueden unir las tablas sin duplicar datos:

- `DATA`: **una fila por reporte** → `VAERS_ID` debería ser único.
- `VAX`: **una fila por vacuna** → un reporte puede tener varias (relación 1:N).
- `SYMPTOMS`: hasta 5 síntomas por fila, puede haber varias filas por reporte.

In [ ]:
df_data.info()


In [ ]:
# Verificación del grano: ¿es VAERS_ID único en cada archivo?
for nombre, df in [("DATA", df_data), ("VAX", df_vax), ("SYMPTOMS", df_symptoms)]:
    n, u = len(df), df["VAERS_ID"].nunique()
    estado = "único (1 fila por reporte)" if n == u else f"NO único ({n-u:,} filas extra: relación 1:N)"
    print(f"{nombre:9s}: {n:>8,} filas / {u:>8,} VAERS_ID distintos  ->  {estado}")


## 4. Calidad: tasa de nulos

Regla práctica (framework de perfilado): >5% de nulos → revisar; >20% → alerta. No todo nulo
es un problema, pero cada uno merece una mirada.

In [ ]:
def tasa_nulos(df, top=15):
    t = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    return t.head(top).rename("% nulos").to_frame()

print("Top columnas con más nulos en DATA:")
tasa_nulos(df_data)


## 5. Aislar los reportes de vacunas COVID-19

En `VAX`, la columna `VAX_TYPE` identifica el tipo de vacuna. Los reportes COVID tienen
`VAX_TYPE == "COVID19"`. Filtramos ahí y recuperamos los `VAERS_ID` correspondientes.

In [ ]:
print("Tipos de vacuna más frecuentes en VAX:")
print(df_vax["VAX_TYPE"].value_counts().head(10), "\n")

vax_covid = df_vax[df_vax["VAX_TYPE"] == "COVID19"].copy()
ids_covid = vax_covid["VAERS_ID"].unique()
print(f"Reportes con al menos una vacuna COVID-19: {len(ids_covid):,}")

# Reportes COVID en la tabla DATA (deduplicados por VAERS_ID)
data_covid = df_data[df_data["VAERS_ID"].isin(ids_covid)].drop_duplicates("VAERS_ID")
print(f"Filas en DATA para esos reportes: {len(data_covid):,}")


## 6. El conteo crudo — la lectura *ingenua*

Este es exactamente el número que circula mal interpretado. Lo mostramos **para luego
corregirlo**, no como conclusión.

⚠️ Un conteo de reportes **no** es un conteo de casos causados por la vacuna. Sin denominador
(dosis administradas) este número no dice nada sobre riesgo.

In [ ]:
n_reportes = len(data_covid)
n_muertes  = (data_covid["DIED"] == "Y").sum()
n_hosp     = (data_covid["HOSPITAL"] == "Y").sum()

print("LECTURA CRUDA (NO interpretar como causalidad):")
print(f"  Reportes COVID-19 totales : {n_reportes:,}")
print(f"  Con DIED == 'Y'           : {n_muertes:,}")
print(f"  Con HOSPITAL == 'Y'       : {n_hosp:,}")
print("\nEstos números por sí solos son engañosos: faltan el denominador y la causalidad.")


## 7. Traer el denominador (dosis administradas)

Aquí sí funciona la descarga directa: Our World in Data publica las dosis administradas en
EE. UU., que es el denominador que VAERS no trae. Este es el puente hacia el análisis de
**tasas por millón de dosis** que haremos en el siguiente notebook.

In [ ]:
url_owid = ("https://raw.githubusercontent.com/owid/covid-19-data/master/"
            "public/data/vaccinations/vaccinations.csv")
vac = pd.read_csv(url_owid)
us = vac[vac["location"] == "United States"].copy()
us["date"] = pd.to_datetime(us["date"])

dosis_totales_us = us["total_vaccinations"].max()
print(f"Dosis COVID-19 administradas en EE. UU. (máximo acumulado OWID): {dosis_totales_us:,.0f}")

# Ilustración del cambio de lectura (aún preliminar, cobertura temporal no ajustada):
tasa = n_reportes / dosis_totales_us * 1_000_000
print(f"\nReportes por millón de dosis (preliminar): {tasa:,.1f}")
print("Ojo: comparar rangos de fechas VAERS vs OWID es parte del trabajo del próximo notebook.")


## Próximos pasos (notebook 02)

1. **Alinear ventanas temporales** entre reportes VAERS y dosis administradas.
2. **Deduplicar** reportes secundarios (cambio del 8-may-2025) por `VAERS_ID`.
3. Calcular **tasas por millón de dosis** por mes y por tipo de evento (muerte, hospitalización).
4. Contrastar la **lectura cruda vs. la lectura con denominador** — el corazón del proyecto.
5. Redactar hallazgos y limitaciones en el `README.md`.

> Recordatorio permanente: nada aquí establece causalidad. VAERS genera *hipótesis*, no
> pruebas; los sistemas de vigilancia activa (VSD, FDA BEST) son los que confirman señales.